CONSTANTS

In [42]:
LOCAL_DIR = "../data/raw/hotel_bookings_with_holidays.csv"
KAGGLE_DIR = "/kaggle/input/datasets/omarhashem80/hotel-booking-demand/hotel_bookings_with_holidays.csv"

## 🔴Import Libraries 📦

In [43]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import chi2_contingency, f_oneway
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import AdaBoostClassifier, HistGradientBoostingClassifier
from sklearn.feature_selection import SelectFromModel
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
import joblib
import warnings
warnings.filterwarnings('ignore')

## 🔴Load Dataset 🗂️

In [44]:
df = pd.read_csv(LOCAL_DIR)

## 🔴Features Understanding 📖

In [45]:
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,arrival_date,is_holiday,days_to_next_holiday,days_from_last_holiday
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,Transient,0.0,0,0,Check-Out,2015-07-01,2015-07-01,1.0,3.0,0.0
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,Transient,0.0,0,0,Check-Out,2015-07-01,2015-07-01,1.0,3.0,0.0
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,Transient,75.0,0,0,Check-Out,2015-07-02,2015-07-01,0.0,11.0,10.0
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,Transient,75.0,0,0,Check-Out,2015-07-02,2015-07-01,0.0,11.0,10.0
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,Transient,98.0,0,1,Check-Out,2015-07-03,2015-07-01,0.0,11.0,10.0


In [46]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 36 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  object 
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  object 
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal            

In [47]:
df.describe()

,is_canceled,lead_time,arrival_date_year,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,...,booking_changes,agent,company,days_in_waiting_list,adr,required_car_parking_spaces,total_of_special_requests,is_holiday,days_to_next_holiday,days_from_last_holiday
count,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119386.000000,119390.000000,...,119390.000000,103050.000000,6797.000000,119390.000000,119390.000000,119390.000000,119390.000000,117614.000000,117614.000000,117613.000000
mean,0.370416,104.011416,2016.156554,27.165173,15.798241,0.927599,2.500302,1.856403,0.103890,0.007949,...,0.221124,86.693382,189.266735,2.321149,101.831122,0.062518,0.571363,0.139167,11.473439,10.628272
std,0.482918,106.863097,0.707476,13.605138,8.780829,0.998613,1.908286,0.579261,0.398561,0.097436,...,0.652306,110.774548,131.655015,17.594721,50.535790,0.245291,0.792798,0.346122,12.283761,12.674977
min,0.000000,0.000000,2015.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,1.000000,6.000000,0.000000,-6.380000,0.000000,0.000000,0.000000,1.000000,0.000000
25%,0.000000,18.000000,2016.000000,16.000000,8.000000,0.000000,1.000000,2.000000,0.000000,0.000000,...,0.000000,9.000000,62.000000,0.000000,69.290000,0.000000,0.000000,0.000000,3.000000,2.000000
50%,0.000000,69.000000,2016.000000,28.000000,16.000000,1.000000,2.000000,2.000000,0.000000,0.000000,...,0.000000,14.000000,179.000000,0.000000,94.575000,0.000000,0.000000,0.000000,7.000000,6.000000
75%,1.000000,160.000000,2017.000000,38.000000,23.000000,2.000000,3.000000,2.000000,0.000000,0.000000,...,0.000000,229.000000,270.000000,0.000000,126.000000,0.000000,1.000000,0.000000,17.000000,15.000000
max,1.000000,737.000000,2017.000000,53.000000,31.000000,19.000000,50.000000,55.000000,10.000000,10.000000,...,21.000000,535.000000,543.000000,391.000000,5400.000000,8.000000,5.000000,1.000000,241.000000,93.000000


In [48]:
df.describe(include='object')

,hotel,arrival_date_month,meal,country,market_segment,distribution_channel,reserved_room_type,assigned_room_type,deposit_type,customer_type,reservation_status,reservation_status_date,arrival_date
count,119390,119390,119390,118902,119390,119390,119390,119390,119390,119390,119390,119390,119390
unique,2,12,5,177,8,5,10,12,3,4,3,926,793
top,City Hotel,August,BB,PRT,Online TA,TA/TO,A,A,No Deposit,Transient,Check-Out,2015-10-21,2015-12-05
freq,79330,13877,92310,48590,56477,97870,85994,74053,104641,89613,75166,1461,448


In [49]:
for col in df.select_dtypes(include='object').columns:
  print(f"num of unique values: {df[col].nunique()}")
  print(df[col].value_counts(normalize=True) * 100,"*"*50,sep='\n')

num of unique values: 2
hotel
City Hotel      66.446101
Resort Hotel    33.553899
Name: proportion, dtype: float64
**************************************************
num of unique values: 12
arrival_date_month
August       11.623252
July         10.604741
May           9.876037
October       9.347517
April         9.288048
June          9.162409
September     8.801407
March         8.203367
February      6.757685
November      5.690594
December      5.678868
January       4.966078
Name: proportion, dtype: float64
**************************************************
num of unique values: 5
meal
BB           77.318033
HB           12.114080
SC            8.920345
Undefined     0.979144
FB            0.668398
Name: proportion, dtype: float64
**************************************************
num of unique values: 177
country
PRT    40.865587
GBR    10.200838
FRA     8.759314
ESP     7.205934
DEU     6.128576
         ...    
MRT     0.000841
KIR     0.000841
SDN     0.000841
ATF     0.00084

In [50]:
top_k = 5

top_countries = df['country'].value_counts().head(top_k).index

df.country.value_counts(normalize=True) * 100

country
PRT    40.865587
GBR    10.200838
FRA     8.759314
ESP     7.205934
DEU     6.128576
         ...    
MRT     0.000841
KIR     0.000841
SDN     0.000841
ATF     0.000841
SLE     0.000841
Name: proportion, Length: 177, dtype: float64

In [51]:
df['country'] = df['country'].apply(
    lambda x: x if x in top_countries else 'Other'
)

In [52]:
top_agents = df['agent'].value_counts().head(10).index
top_agents

Index([9.0, 240.0, 1.0, 14.0, 7.0, 6.0, 250.0, 241.0, 28.0, 8.0], dtype='float64', name='agent')

In [53]:
df['agent'] = df['agent'].apply(
    lambda x: x if x in top_agents else 'Other'
)

In [54]:
df.agent.value_counts(normalize=True) * 100

agent
Other    40.268029
9.0      26.770249
240.0    11.660943
1.0       6.023118
14.0      3.048832
7.0       2.964235
6.0       2.755675
250.0     2.403886
241.0     1.441494
28.0      1.395427
8.0       1.268113
Name: proportion, dtype: float64

## 🔴Cleaning🕵️

In [55]:
df.isna().mean() * 100

hotel                              0.000000
is_canceled                        0.000000
lead_time                          0.000000
arrival_date_year                  0.000000
arrival_date_month                 0.000000
arrival_date_week_number           0.000000
arrival_date_day_of_month          0.000000
stays_in_weekend_nights            0.000000
stays_in_week_nights               0.000000
adults                             0.000000
children                           0.003350
babies                             0.000000
meal                               0.000000
country                            0.000000
market_segment                     0.000000
distribution_channel               0.000000
is_repeated_guest                  0.000000
previous_cancellations             0.000000
previous_bookings_not_canceled     0.000000
reserved_room_type                 0.000000
assigned_room_type                 0.000000
booking_changes                    0.000000
deposit_type                    

In [56]:
df["children"].fillna(0, inplace=True)
df["country"].fillna("Unknown", inplace=True)
df["agent"].fillna(0, inplace=True)
df["is_holiday"].fillna(0, inplace=True)
df["days_to_next_holiday"].fillna(-1, inplace=True)
df["days_from_last_holiday"].fillna(-1, inplace=True)

In [57]:
df["reservation_status_date"] = pd.to_datetime(df["reservation_status_date"])
df['reservation_status_year'] = df['reservation_status_date'].dt.year
df['reservation_status_month'] = df['reservation_status_date'].dt.month
df['reservation_status_day'] = df['reservation_status_date'].dt.day
df["arrival_date"] = pd.to_datetime(df["arrival_date"])
df['arrival_date_year'] = df['arrival_date'].dt.year
df['arrival_date_month'] = df['arrival_date'].dt.month
df['arrival_date_week_number'] = df['arrival_date'].dt.isocalendar().week
df['arrival_date_day_of_month'] = df['arrival_date'].dt.day

In [58]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 39 columns):
 #   Column                          Non-Null Count   Dtype         
---  ------                          --------------   -----         
 0   hotel                           119390 non-null  object        
 1   is_canceled                     119390 non-null  int64         
 2   lead_time                       119390 non-null  int64         
 3   arrival_date_year               119390 non-null  int32         
 4   arrival_date_month              119390 non-null  int32         
 5   arrival_date_week_number        119390 non-null  UInt32        
 6   arrival_date_day_of_month       119390 non-null  int32         
 7   stays_in_weekend_nights         119390 non-null  int64         
 8   stays_in_week_nights            119390 non-null  int64         
 9   adults                          119390 non-null  int64         
 10  children                        119390 non-null  float64

In [59]:
cat_cols = [
    'hotel',
    'arrival_date_month',
    'meal',
    'country',
    'market_segment',
    'distribution_channel',
    'reserved_room_type',
    'assigned_room_type',
    'deposit_type',
    'customer_type',
    'reservation_status',
    'arrival_date_year',
    'arrival_date_week_number',
    'arrival_date_day_of_month',
    'reservation_status_year',
    'reservation_status_month',
    'reservation_status_day',
    'agent'
]

binary_cols = [
    'is_canceled',
    'is_repeated_guest',
    'is_holiday'
]

int_cols = [
    'lead_time',
    'stays_in_weekend_nights',
    'stays_in_week_nights',
    'adults',
    'babies',
    'previous_cancellations',
    'previous_bookings_not_canceled',
    'booking_changes',
    'days_in_waiting_list',
    'required_car_parking_spaces',
    'total_of_special_requests',
    'children',
    'days_to_next_holiday',
    'days_from_last_holiday'
]

float_cols = ['adr']

date_cols = ['arrival_date', 'reservation_status_date']


def clean_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    for col in cat_cols:
        if col in df.columns:
            df[col] = df[col].astype('category')

    for col in binary_cols:
        if col in df.columns:
            df[col] = df[col].astype('int8').astype('category')

    for col in int_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

    for col in float_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').astype('float64')

    for col in date_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')

    return df


df = clean_dtypes(df)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 39 columns):
 #   Column                          Non-Null Count   Dtype         
---  ------                          --------------   -----         
 0   hotel                           119390 non-null  category      
 1   is_canceled                     119390 non-null  category      
 2   lead_time                       119390 non-null  Int64         
 3   arrival_date_year               119390 non-null  category      
 4   arrival_date_month              119390 non-null  category      
 5   arrival_date_week_number        119390 non-null  category      
 6   arrival_date_day_of_month       119390 non-null  category      
 7   stays_in_weekend_nights         119390 non-null  Int64         
 8   stays_in_week_nights            119390 non-null  Int64         
 9   adults                          119390 non-null  Int64         
 10  children                        119390 non-null  Int64  

In [60]:
df.agent = df.agent.astype('str')

In [61]:
df = df[~((df['adults'] == 0) & (df['children'] == 0) & (df['babies'] == 0))]

In [62]:
df.reset_index(drop=True, inplace=True)

## 🔴EDA

In [22]:
nums = df.select_dtypes(include=np.number).columns.tolist()
cats = df.select_dtypes(include='category').columns.tolist()
print("Numerical Columns:", nums)
print("-" * 500)
print("Categorical Columns:", cats)

Numerical Columns: ['lead_time', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'previous_cancellations', 'previous_bookings_not_canceled', 'booking_changes', 'company', 'days_in_waiting_list', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'days_to_next_holiday', 'days_from_last_holiday']
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Categorical Columns: ['hotel', 'is_canceled', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'meal'

In [23]:
nums =  ['lead_time', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'previous_cancellations', 'previous_bookings_not_canceled', 'booking_changes', 'company', 'days_in_waiting_list', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'days_to_next_holiday', 'days_from_last_holiday']
cats = ['hotel', 'is_canceled', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'reserved_room_type', 'assigned_room_type', 'deposit_type', 'customer_type', 'reservation_status', 'is_holiday', 'reservation_status_year', 'reservation_status_month', 'reservation_status_day', 'agent']
dates = ['arrival_date', 'reservation_status_date']

### 🔴Univariante Analysis **👆**

In [ ]:
def create_histogram_with_boxplot(df, x):
    fig = make_subplots(rows=2, cols=1)
    fig.add_trace(go.Histogram(x=df[x], name='Histogram'), row=1, col=1)
    fig.add_trace(go.Box(x=df[x], name="Box"), row=2, col=1)
    fig.update_layout(
        title=f'Distribution of {x}',
        height=600,
    )
    fig.show()

def create_pie_chart(df, cat):
    fig = px.pie(names=df[cat], hole=0.3, title=f"Distribution of {cat}")
    fig.update_traces(textinfo='percent+label')
    fig.update_layout(height=800)
    fig.show()

def create_bar_chart(df, cat, top_n=10):
    dff = df[cat].value_counts().head(top_n).reset_index()
    dff.columns = [cat, 'count']

    fig = px.bar(
        dff,
        x=cat,
        y='count',
        title=f"Top {top_n} {cat}",
        text='count'
    )

    fig.update_layout(
        height=600,
        xaxis_title=cat,
        yaxis_title="Count"
    )

    fig.update_traces(textposition='outside')
    fig.show()

#### 🟢Numerical🔢

In [ ]:
i = 0

In [ ]:
create_histogram_with_boxplot(df, nums[i])
i += 1

In [ ]:
create_histogram_with_boxplot(df, nums[i])
i += 1

In [ ]:
create_histogram_with_boxplot(df, nums[i])
i += 1

In [ ]:
create_histogram_with_boxplot(df, nums[i])
i += 1

In [ ]:
create_histogram_with_boxplot(df, nums[i])
i += 1

In [ ]:
create_histogram_with_boxplot(df, nums[i])
i += 1

In [ ]:
create_histogram_with_boxplot(df, nums[i])
i += 1

In [ ]:
create_histogram_with_boxplot(df, nums[i])
i += 1

In [ ]:
create_histogram_with_boxplot(df, nums[i])
i += 1

In [ ]:
create_histogram_with_boxplot(df, nums[i])
i += 1

In [ ]:
create_histogram_with_boxplot(df, nums[i])
i += 1

In [ ]:
create_histogram_with_boxplot(df, nums[i])
i += 1

In [ ]:
create_histogram_with_boxplot(df, nums[i])
i += 1

In [ ]:
create_histogram_with_boxplot(df, nums[i])
i += 1

In [ ]:
create_histogram_with_boxplot(df, nums[i])
i += 1

In [ ]:
create_histogram_with_boxplot(df, nums[i])
i += 1

In [ ]:
assert i == len(nums), "Not all numerical columns were plotted for cancellation analysis."

#### 🔵Categorical

In [ ]:
i = 0

In [ ]:
create_pie_chart(df, cats[i])
i += 1

In [ ]:
create_pie_chart(df, cats[i])
i += 1

In [ ]:
create_pie_chart(df, cats[i])
i += 1

In [ ]:
create_pie_chart(df, cats[i])
i += 1

In [ ]:
create_bar_chart(df, 'arrival_date_week_number')
i += 1

In [ ]:
create_pie_chart(df, cats[i])
i += 1

In [ ]:
create_pie_chart(df, cats[i])
i += 1

In [ ]:
create_pie_chart(df, cats[i])
i += 1

In [ ]:
country_data = df.groupby("country")["adults"].sum().sort_values(ascending=False).head(20)
country_data = (country_data / country_data.sum()) * 100

In [ ]:
# Aggregate
country_data = (
    df.groupby("country")["adults"]
    .sum()
    .sort_values(ascending=False)
    .head(20)
)

# Convert to percentage
country_data = (country_data / country_data.sum()) * 100

# Convert Series → DataFrame (IMPORTANT FIX)
country_data = country_data.reset_index()
country_data.columns = ["country", "Guests in %"]

# Plot
guest_map = px.choropleth(
    country_data,
    locations="country",
    locationmode="ISO-3",
    color="Guests in %",
    hover_name="country",
    color_continuous_scale="Plasma",
    title="Home Country of Guests"
)

guest_map.show()

In [ ]:
create_pie_chart(df, cats[i])
i += 1

In [ ]:
create_pie_chart(df, cats[i])
i += 1

In [ ]:
create_pie_chart(df, cats[i])
i +=1

In [ ]:
create_pie_chart(df, cats[i])
i += 1

In [ ]:
create_pie_chart(df, cats[i])
i += 1

In [ ]:
create_pie_chart(df, cats[i])
i += 1

In [ ]:
# replace the plot
create_bar_chart(df, 'agent', top_n=10)

In [ ]:
create_pie_chart(df, cats[i])
i += 1

In [ ]:
create_pie_chart(df, cats[i])
i += 1

In [ ]:
create_pie_chart(df, cats[i])
i += 1

In [ ]:
create_pie_chart(df, cats[i])
i += 1

In [ ]:
create_pie_chart(df, cats[i])
i += 1

In [ ]:
create_pie_chart(df, cats[i])
i += 1

In [ ]:
assert i + 1 == len(cats), "Not all categorical columns were plotted."

### Dates

In [ ]:
def plot_time_series(df, date_col, year):
    dff = df[df[date_col].dt.year == year]
    dff = (
        dff.groupby(date_col)
        .size()
        .reset_index(name='count')
        .sort_values(date_col)
    )

    fig = px.line(
        dff,
        x=date_col,
        y='count'
    )

    fig.show()

In [ ]:
plot_time_series(df, 'arrival_date', 2015)

In [ ]:
plot_time_series(df, 'arrival_date', 2016)

In [ ]:
plot_time_series(df, 'arrival_date', 2017)

In [ ]:
plot_time_series(df, 'reservation_status_date', 2017)

In [ ]:
plot_time_series(df, 'reservation_status_date', 2016)

In [ ]:
plot_time_series(df, 'reservation_status_date', 2015)

In [ ]:
def plot_years_overlay(df, date_col='arrival_date'):
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])

    df['year'] = df[date_col].dt.year
    df['day_of_year'] = df[date_col].dt.dayofyear

    dff = (
        df.groupby(['year', 'day_of_year'])
        .size()
        .reset_index(name='count')
    )

    fig = px.line(
        dff,
        x='day_of_year',
        y='count',
        color='year',
        title="Yearly Comparison (Overlay)"
    )

    fig.update_layout(
        xaxis_title="Day of Year",
        yaxis_title="Bookings"
    )

    fig.show()

In [ ]:
plot_years_overlay(df, 'arrival_date')

### 🔴Bivariante Analysis✌️

#### 🟢Numerical vs Numerical

In [ ]:
px.imshow(df[nums].corr(numeric_only=True, method='spearman'), text_auto='.2f', width=1000, height=800)

#### ⚫Categorical vs Numerical


In [ ]:
def plot_mean_num_by_cancellation(df, x):
    dff = df.groupby('is_canceled')[x].mean().reset_index()
    fig = px.bar(dff, x='is_canceled', y=x, title='Mean ' + x.replace('_', ' ').title() + ' by Cancellation Status',
             labels={'is_canceled': 'Is Canceled', x: 'Mean ' + x.replace('_', ' ').title()}, text_auto='.2f' )
    fig.update_layout(
        xaxis={'categoryorder': 'total ascending'}
    )

    fig.show()

In [ ]:
nums

In [ ]:
i = 0

In [ ]:
plot_mean_num_by_cancellation(df, nums[i])
i += 1

In [ ]:
plot_mean_num_by_cancellation(df, nums[i])
i += 1

In [ ]:
plot_mean_num_by_cancellation(df, nums[i])
i += 1

In [ ]:
plot_mean_num_by_cancellation(df, nums[i])
i += 1

In [ ]:
plot_mean_num_by_cancellation(df, nums[i])
i += 1

In [ ]:
plot_mean_num_by_cancellation(df, nums[i])
i += 1

In [ ]:
plot_mean_num_by_cancellation(df, nums[i])
i += 1

In [ ]:
plot_mean_num_by_cancellation(df, nums[i])
i += 1

In [ ]:
plot_mean_num_by_cancellation(df, nums[i])
i += 1

In [ ]:
# WRONG TODO: fix it
plot_mean_num_by_cancellation(df, nums[i])
i += 1

In [ ]:
plot_mean_num_by_cancellation(df, nums[i])
i += 1

In [ ]:
plot_mean_num_by_cancellation(df, nums[i])
i += 1

In [ ]:
plot_mean_num_by_cancellation(df, nums[i])
i += 1

In [ ]:
plot_mean_num_by_cancellation(df, nums[i])
i += 1

In [ ]:
plot_mean_num_by_cancellation(df, nums[i])
i += 1

In [ ]:
plot_mean_num_by_cancellation(df, nums[i])
i += 1

In [ ]:
assert i == len(nums), "Not all numerical columns were plotted for cancellation analysis."

In [ ]:
def eta_squared(groups):
    all_values = np.concatenate(groups)
    grand_mean = np.mean(all_values)
    ss_between = sum(len(g) * (np.mean(g) - grand_mean)**2 for g in groups)
    ss_total = sum((x - grand_mean)**2 for x in all_values)
    return ss_between / ss_total

def interpret_eta_squared(eta):
    if eta < 0.01:
        return "Negligible"
    elif eta < 0.06:
        return "Weak"
    elif eta < 0.14:
        return "Moderate"
    else:
        return "Strong"


alpha = 0.05
for num in nums:
    groups = [df[df['is_canceled'] == cat][num].dropna() for cat in df['is_canceled'].unique()]
    f_stat, p = f_oneway(*groups)
    eta = eta_squared(groups)
    strength = interpret_eta_squared(eta)

    print(f"Feature: {num}")
    print(f"F-statistic: {f_stat:.2f}, p-value: {p:.6f}, Eta²: {eta:.3f}")
    print(f"Association strength: {strength}")

    if p < alpha:
        print("Significant association")
    else:
        print("No significant association")
    print("-" * 50)

#### ⚫Categorical vs Categorical

In [ ]:
def create_bar_chart(df, cat):
    dff = df.groupby(cat)['is_canceled'].value_counts(normalize=True).unstack().reset_index()
    dff = dff.melt(id_vars=cat, var_name='is_canceled', value_name='proportion')
    print(dff)
    fig = px.bar(dff, x=cat, y='proportion', color='is_canceled', barmode='group', text_auto='.2f',
             title=f'Distribution of is_canceled by {cat}', labels={cat: cat.capitalize(), 'proportion': 'Proportion', 'is_canceled': 'Is Canceled'})
    fig.show()

In [ ]:
i = 0

In [ ]:
create_bar_chart(df, cats[i])
i += 1

In [ ]:
i += 1
create_bar_chart(df, cats[i])
i += 1

In [ ]:
create_bar_chart(df, cats[i])
i += 1

In [ ]:
#TODO: replace the plot
create_bar_chart(df, cats[i])
i += 1

In [ ]:
create_bar_chart(df, cats[i])
i += 1

In [ ]:
create_bar_chart(df, cats[i])
i += 1

In [ ]:
#TODO: replace the plot
create_bar_chart(df, cats[i])
i += 1

In [ ]:
create_bar_chart(df, cats[i])
i += 1

In [ ]:
create_bar_chart(df, cats[i])
i += 1

In [ ]:
create_bar_chart(df, cats[i])
i += 1

In [ ]:
create_bar_chart(df, cats[i])
i += 1

In [ ]:
create_bar_chart(df, cats[i])
i += 1

In [ ]:
create_bar_chart(df, cats[i])
i += 1

In [ ]:
create_bar_chart(df, cats[i])
i += 1

In [ ]:
create_bar_chart(df, cats[i])
i += 1

In [ ]:
create_bar_chart(df, cats[i])
i += 1

In [ ]:
create_bar_chart(df, cats[i])
i += 1

In [ ]:
create_bar_chart(df, cats[i])
i += 1

In [ ]:
create_bar_chart(df, cats[i])
i += 1

In [ ]:
i += 1

In [ ]:
assert i == len(cats), "Not all categorical columns were plotted."

In [ ]:
def cramers_v(table):
    chi2 = chi2_contingency(table)[0]
    n = table.sum().sum()
    r, k = table.shape
    return np.sqrt(chi2 / (n * (min(r, k) - 1)))


def interpret_cramers_v(v):
    if v < 0.1:
        return "Negligible"
    elif v < 0.3:
        return "Weak"
    elif v < 0.5:
        return "Moderate"
    else:
        return "Strong"


alpha = 0.05
C_cats = [cat for cat in cats if cat != 'is_canceled']
for cat in C_cats:
    table = pd.crosstab(df[cat], df['is_canceled'])
    chi2, p, dof, _ = chi2_contingency(table)
    v = cramers_v(table)
    strength = interpret_cramers_v(v)

    print(f"\nFeature: {cat}")
    print(f"Chi-square: {chi2:.2f}, p-value: {p:.6f}, Cramér's V: {v:.3f}")
    print(f"Association strength: {strength}")

    if p < alpha:
        print("Significant association")
    else:
        print("No significant association")

### date vs cat

In [ ]:
def plot_years_overlay_canceled(df, date_col='arrival_date'):
    df = df[df['is_canceled'] == 1] .copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df['year'] = df[date_col].dt.year
    df['day_of_year'] = df[date_col].dt.dayofyear

    dff = df.groupby(['year', 'day_of_year']).size().reset_index(name='count')

    fig = px.line(
        dff,
        x='day_of_year',
        y='count',
        color='year',
        title="Yearly Comparison (Overlay)"
    )

    fig.update_layout(
        xaxis_title="Day of Year",
        yaxis_title="Bookings"
    )

    fig.show()

In [ ]:
plot_years_overlay_canceled(df, 'arrival_date')

In [ ]:
plot_years_overlay_canceled(df, 'reservation_status_date')

In [ ]:
def plot_time_series(df, date_col, year):
    dff = df.copy()
    dff[date_col] = pd.to_datetime(dff[date_col])
    dff = dff[dff[date_col].dt.year == year]

    dff = dff.groupby([date_col, 'is_canceled']).size().reset_index(name='count').sort_values(date_col)
    dff['status'] = dff['is_canceled'].map({0: 'Not Canceled', 1: 'Canceled'})

    fig = px.line(
        dff,
        x=date_col,
        y='count',
        color='status',
    )

    fig.show()

In [ ]:
plot_time_series(df, 'arrival_date', 2017)

In [ ]:
plot_time_series(df, 'arrival_date', 2016)

In [ ]:
plot_time_series(df, 'arrival_date', 2015)

In [ ]:
def plot_all_days_trend(df, date_col):
    dff = df.copy()
    dff[date_col] = pd.to_datetime(dff[date_col])
    dff['day_of_year'] = dff[date_col].dt.dayofyear

    dff = dff.groupby(['day_of_year', 'is_canceled']).size().reset_index(name='count').sort_values('day_of_year')
    dff['status'] = dff['is_canceled'].map({0: 'Not Canceled', 1: 'Canceled'})

    fig = px.line(
        dff,
        x='day_of_year',
        y='count',
        color='status'
    )

    fig.show()

In [ ]:
plot_all_days_trend(df, 'arrival_date')

In [ ]:
plot_all_days_trend(df, 'reservation_status_date')

## 🔴Preprocessing

In [63]:
df.drop("reservation_status", axis=1, inplace=True)

In [64]:
cat_cols = [
    'hotel',
    'meal',
    'country',
    'market_segment',
    'distribution_channel',
    'reserved_room_type',
    'assigned_room_type',
    'deposit_type',
    'customer_type',
    'agent'
]

binary_cols = [
    'is_repeated_guest',
    'is_holiday'
]

target_col = 'is_canceled'

int_cols = [
    'arrival_date_year',
    'arrival_date_week_number',
    'arrival_date_day_of_month',
    'arrival_date_month',
    'lead_time',
    'stays_in_weekend_nights',
    'stays_in_week_nights',
    'adults',
    'children',
    'babies',
    'previous_cancellations',
    'previous_bookings_not_canceled',
    'booking_changes',
    'days_in_waiting_list',
    'required_car_parking_spaces',
    'total_of_special_requests',
    'days_to_next_holiday',
    'days_from_last_holiday',
    'reservation_status_year',
    'reservation_status_month',
    'reservation_status_day'
]

float_cols = ['adr']

date_cols = ['arrival_date']

In [65]:
df['is_canceled'] = df['is_canceled'].astype('int8')
for col in binary_cols:
    if col in df.columns:
        df[col] = df[col].astype('int8')

for col in int_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

In [66]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119210 entries, 0 to 119209
Data columns (total 38 columns):
 #   Column                          Non-Null Count   Dtype         
---  ------                          --------------   -----         
 0   hotel                           119210 non-null  category      
 1   is_canceled                     119210 non-null  int8          
 2   lead_time                       119210 non-null  Int64         
 3   arrival_date_year               119210 non-null  Int64         
 4   arrival_date_month              119210 non-null  Int64         
 5   arrival_date_week_number        119210 non-null  Int64         
 6   arrival_date_day_of_month       119210 non-null  Int64         
 7   stays_in_weekend_nights         119210 non-null  Int64         
 8   stays_in_week_nights            119210 non-null  Int64         
 9   adults                          119210 non-null  Int64         
 10  children                        119210 non-null  Int64  

In [ ]:
# def month_cyclical(X):
#     month = X[:, 0]

#     month = pd.to_numeric(month, errors='coerce')
#     month = np.nan_to_num(month, nan=0)
#     month = month.astype(float)

#     return np.column_stack([
#         np.sin(2 * np.pi * month / 12),
#         np.cos(2 * np.pi * month / 12)
#     ])

In [67]:
numeric_cols = [
    'lead_time',
    'stays_in_weekend_nights',
    'stays_in_week_nights',
    'adults',
    'children',
    'babies',
    'previous_cancellations',
    'previous_bookings_not_canceled',
    'booking_changes',
    'days_in_waiting_list',
    'adr',
    'required_car_parking_spaces',
    'total_of_special_requests',
    'days_to_next_holiday',
    'days_from_last_holiday',
    'arrival_date_year',
    'arrival_date_week_number',
    'arrival_date_day_of_month',
    'reservation_status_year',
    'reservation_status_day'
]

categorical_cols = [
    'hotel',
    'meal',
    'country',
    'market_segment',
    'distribution_channel',
    'reserved_room_type',
    'assigned_room_type',
    'deposit_type',
    'customer_type',
    'agent',
    'is_repeated_guest',
    'is_holiday'
]

month_col = ['arrival_date_month', 'reservation_status_month']

target_col = 'is_canceled'

In [29]:
# df.drop('country', axis=1, inplace=True)

In [30]:
def build_preprocessor():

    numeric_pipeline = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    categorical_pipeline = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])

    month_pipeline = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        # ('cyclical', FunctionTransformer(month_cyclical, feature_names_out=lambda self, input_features=None: ['month_sin', 'month_cos']))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_pipeline, numeric_cols),
            ('cat', categorical_pipeline, categorical_cols),
            ('month', month_pipeline, month_col)
        ],
        remainder='drop'
    )

    return preprocessor

In [68]:
df.duplicated().sum()

np.int64(32026)

In [69]:
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

In [ ]:
# df.to_csv('../data/processed/hotel_bookings.csv', index=False)

In [ ]:
df.info()

In [ ]:
pdf

In [ ]:
df.info()

In [ ]:
def group_rare_categories(df, col, min_freq=100):
    counts = df[col].value_counts()
    rare = counts[counts < min_freq].index
    df[col] = df[col].replace(rare, "Other")
    return df

In [ ]:
for col in df.select_dtypes(include=["object", "category"]).columns:    
    df = group_rare_categories(df, col, min_freq=900)

In [33]:
for col in df.select_dtypes(include=["object", "category"]).columns:
  print(f"num of unique values: {df[col].nunique()}")
  print(df[col].value_counts(normalize=True) * 100,"*"*50,sep='\n')

num of unique values: 2
hotel
City Hotel      61.068545
Resort Hotel    38.931455
Name: proportion, dtype: float64
**************************************************
num of unique values: 5
meal
BB           77.844559
SC           10.769178
HB           10.410167
Undefined     0.564324
FB            0.411773
Name: proportion, dtype: float64
**************************************************
num of unique values: 6
country
Other    32.072399
PRT      31.368141
GBR      11.956322
FRA      10.119976
ESP       8.307717
DEU       6.175445
Name: proportion, dtype: float64
**************************************************
num of unique values: 8
market_segment
Online TA        59.119793
Offline TA/TO    15.872178
Direct           13.510507
Groups            5.628326
Corporate         4.813957
Complementary     0.793724
Aviation          0.259222
Undefined         0.002294
Name: proportion, dtype: float64
**************************************************
num of unique values: 5
distribution_

In [ ]:
X = df.drop("is_canceled", axis=1)
y = df.is_canceled

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
def run_model_pipeline(
    name,
    model,
    param_grid,
    X_train,
    y_train,
    X_test,
    y_test,
    selector=None,
    sampler=None
):

    print(f"\n{'='*30} {name} {'='*30}")

    if sampler is not None:
        PipelineClass = ImbPipeline
    else:
        PipelineClass = Pipeline

    steps = []

    steps.append(('preprocessing', build_preprocessor()))

    if selector is not None:
        steps.append(('feature_selection', selector))

    if sampler is not None:
        steps.append(('sampler', sampler))

    steps.append(('model', model))

    pipeline = PipelineClass(steps=steps)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    fixed_param_grid = {
        f"model__{key}": value
        for key, value in param_grid.items()
    }

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=fixed_param_grid,
        cv=cv,
        scoring='f1',
        n_jobs=-1,
        return_train_score=True,
        error_score='raise'
    )

    search.fit(X_train, y_train)

    print(f"\nBest CV F1: {search.best_score_:.4f}")
    print(f"Best Params: {search.best_params_}")

    best_model = search.best_estimator_

    y_train_pred = best_model.predict(X_train)
    y_test_pred = best_model.predict(X_test)

    target_names = ['Not Canceled', 'Canceled']

    print("\nTrain Report:")
    print(classification_report(y_train, y_train_pred, target_names=target_names))

    print("\nTest Report:")
    print(classification_report(y_test, y_test_pred, target_names=target_names))

    return search.best_score_, best_model

In [ ]:
sampler = SMOTE(random_state=42)


selector = SelectFromModel(
    estimator=XGBClassifier(
        n_estimators=100,
        random_state=42,
        eval_metric='logloss'
    ),
    threshold='median'
)

In [ ]:
# sampler = None

In [ ]:
results = {}

results['LR'] = run_model_pipeline(
    "Logistic Regression",
    LogisticRegression(max_iter=2000, random_state=42),
    {
        'C': [0.01, 0.1, 1, 10]
    },
    X_train, y_train, X_test, y_test,
    selector=selector,
    sampler=sampler
)


results['XGB'] = run_model_pipeline(
    "XGBoost",
    XGBClassifier(
        random_state=42,
        eval_metric='logloss',
        n_jobs=-1
    ),
    {
        'n_estimators': [100, 200],
        'max_depth': [4, 6, 8],
        'learning_rate': [0.05, 0.1]
    },
    X_train, y_train, X_test, y_test,
    selector=selector,
    sampler=sampler
)


results['CatBoost'] = run_model_pipeline(
    "CatBoost",
    CatBoostClassifier(
        verbose=0,
        random_state=42
    ),
    {
        'iterations': [200, 400],
        'depth': [4, 6, 8],
        'learning_rate': [0.05, 0.1]
    },
    X_train, y_train, X_test, y_test,
    selector=selector,
    sampler=sampler
)

results['AdaBoost'] = run_model_pipeline(
    "AdaBoost",
    AdaBoostClassifier(random_state=42),
    {
        'n_estimators': [50, 100, 200],
        'learning_rate': [0.1, 0.5, 1.0]
    },
    X_train, y_train, X_test, y_test,
    selector=selector,
    sampler=sampler
)

results['HGB'] = run_model_pipeline(
    "HistGradientBoosting",
    HistGradientBoostingClassifier(random_state=42),
    {
        'max_depth': [3, 6, 10],
        'learning_rate': [0.05, 0.1],
        'max_iter': [100, 200]
    },
    X_train, y_train, X_test, y_test,
    selector=selector,
    sampler=sampler
)

In [ ]:
best_name = 'XGB'
best_score, best_model = results[best_name]

print(f"FINAL BEST MODEL: {best_name}")
print(f"Score: {best_score:.4f}")

params = best_model.get_params()

filtered_params = {
    k: v for k, v in params.items()
    if "model__" in k or k in ["model"]
}
print("\nBest Model Params:")
for k, v in filtered_params.items():
    print(f"{k}: {v}")

In [ ]:
best_model = results['XGB'][1]

preprocessor = best_model.named_steps['preprocessing']
selector = best_model.named_steps['feature_selection']
feature_names = preprocessor.get_feature_names_out()
mask = selector.get_support()
selected_features = feature_names[mask]
print(selected_features)

In [ ]:
cm = confusion_matrix(y_test, best_model.predict(X_test))
cm_df = pd.DataFrame(
    cm,
    index=["Actual: Neutral/Dissatisfied", "Actual: Satisfied"],
    columns=["Predicted: Neutral/Dissatisfied", "Predicted: Satisfied"]
)

fig = px.imshow(
    cm_df,
    text_auto=True,
    color_continuous_scale="Blues",
    title="Confusion Matrix (Test Set)"
)

fig.update_layout(
    xaxis_title="Predicted Label",
    yaxis_title="Actual Label"
)

fig.show()

In [ ]:
model_step = best_model.named_steps['model']

preprocessor = best_model.named_steps['preprocessing']
selector = best_model.named_steps['feature_selection']
all_features = preprocessor.get_feature_names_out()
mask = selector.get_support()
selected_features = all_features[mask]
importances = model_step.feature_importances_ * 100
importance_series = pd.Series(importances, index=selected_features).sort_values(ascending=False)

importance_series

In [ ]:
joblib.dump(best_model, "best_model.pkl")

In [ ]:
model = joblib.load('../reports/best_model.pkl')

In [ ]:
model

In [ ]:
numerical_cols = ['lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'booking_changes', 'company', 'days_in_waiting_list', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'is_holiday', 'days_to_next_holiday', 'days_from_last_holiday', 'reservation_status_year', 'reservation_status_month', 'reservation_status_day']
numerical_cols

In [ ]:
categorical_cols = ['hotel', 'meal', 'country', 'market_segment', 'distribution_channel', 'reserved_room_type', 'assigned_room_type', 'deposit_type', 'agent', 'customer_type', 'reservation_status_date', 'arrival_date']
categorical_cols

In [36]:
pdf = pd.read_csv('../data/processed/hotel_bookings_processed.csv', parse_dates=["arrival_date", "reservation_status_date"])

In [38]:
pdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 87175 entries, 0 to 87174
Data columns (total 37 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   hotel                           87175 non-null  object        
 1   is_canceled                     87175 non-null  int64         
 2   lead_time                       87175 non-null  int64         
 3   arrival_date_year               87175 non-null  int64         
 4   arrival_date_month              87175 non-null  int64         
 5   arrival_date_week_number        87175 non-null  int64         
 6   arrival_date_day_of_month       87175 non-null  int64         
 7   stays_in_weekend_nights         87175 non-null  int64         
 8   stays_in_week_nights            87175 non-null  int64         
 9   adults                          87175 non-null  int64         
 10  children                        87175 non-null  int64         
 11  ba

In [78]:
df.equals(pdf)

False

In [72]:
df.drop('country', inplace=True, axis=1)

In [74]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 87184 entries, 0 to 87183
Data columns (total 37 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   hotel                           87184 non-null  category      
 1   is_canceled                     87184 non-null  int8          
 2   lead_time                       87184 non-null  Int64         
 3   arrival_date_year               87184 non-null  Int64         
 4   arrival_date_month              87184 non-null  Int64         
 5   arrival_date_week_number        87184 non-null  Int64         
 6   arrival_date_day_of_month       87184 non-null  Int64         
 7   stays_in_weekend_nights         87184 non-null  Int64         
 8   stays_in_week_nights            87184 non-null  Int64         
 9   adults                          87184 non-null  Int64         
 10  children                        87184 non-null  Int64         
 11  ba

In [77]:
df = df.astype(str)
pdf = pdf.astype(str)